# ACMC1 — frozen external-source evaluation on Adrian validation

Mengevaluasi tiga pasangan checkpoint D0FT/ACMC1 yang **sudah dibekukan oleh locked test Faruq-v3** pada validation Adrian nyata. Tidak ada training, tidak membuka test A0, tidak memilih ulang checkpoint, dan hasil tidak mengubah kesimpulan locked-test `NOT_CONFIRMED`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json, os, shutil, subprocess, sys, time
from pathlib import Path
REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(1, 4):
    result = subprocess.run(clone)
    if result.returncode == 0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt == 3: raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)

In [ ]:
import torch
from coffee_detector.archive_sni21_pilot import restore_real_a0_validation
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
REQUIRED = (
    'bundles/sni21-vadcp-pilot-bundle/A0_real.tar',
    'evidence/faruq-grouped-development-v1/faruq_grouped_manifest.json',
    'experiments/faruq-v3-acmc-locked-test-v2/faruq_v3_acmc_locked_test_summary.json',
    'experiments/faruq-v3-acmc-optimization-control-v1/D0FT_seed42/weights/best.pt',
    'experiments/faruq-v3-acmc-paired-confirmation-v1/D0FT/D0FT_seed123/weights/best.pt',
    'experiments/faruq-v3-acmc-paired-confirmation-v1/D0FT/D0FT_seed2026/weights/best.pt',
    'experiments/faruq-v3-acmc-one-stage-v1/ACMC1_seed42/weights/best.pt',
    'experiments/faruq-v3-acmc-paired-confirmation-v1/ACMC1/ACMC1_seed123/weights/best.pt',
    'experiments/faruq-v3-acmc-paired-confirmation-v1/ACMC1/ACMC1_seed2026/weights/best.pt',
)
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=REQUIRED)
artifacts = {item: require_project_artifact(PROJECT_ROOT, item) for item in REQUIRED}
A0_ROOT = restore_real_a0_validation(artifacts[REQUIRED[0]], '/content/sni21-a0-adrian-external-source')
assert not (A0_ROOT / 'test').exists(), 'Test A0 tidak boleh tersedia.'
ADRIAN_ROOT = Path('/content/sni21-adrian-external-val')
OUTPUT_ROOT = PROJECT_ROOT / 'experiments/faruq-v3-acmc-adrian-external-v1'
print('GPU    :', torch.cuda.get_device_name(0))
print('PROJECT:', PROJECT_ROOT)
print('OUTPUT :', OUTPUT_ROOT)

In [ ]:
command = [
    sys.executable, '-u', '-m', 'coffee_detector.experiments.run_faruq_v3_acmc_adrian_external',
    '--combined-root', str(A0_ROOT),
    '--faruq-manifest', str(artifacts[REQUIRED[1]]),
    '--locked-test-summary', str(artifacts[REQUIRED[2]]),
    '--adrian-root', str(ADRIAN_ROOT), '--output-root', str(OUTPUT_ROOT),
    '--d0ft-checkpoints', *[str(artifacts[item]) for item in REQUIRED[3:6]],
    '--acmc-checkpoints', *[str(artifacts[item]) for item in REQUIRED[6:9]],
    '--seeds', '42', '123', '2026', '--device', '0',
]
LOG = OUTPUT_ROOT / 'adrian_external_run.log'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('MENJALANKAN INFERENCE:', ' '.join(command), flush=True)
process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
with LOG.open('a', encoding='utf-8') as log:
    for line in process.stdout:
        print(line, end='', flush=True); log.write(line); log.flush()
return_code = process.wait()
if return_code != 0:
    print('\n'.join(LOG.read_text(errors='replace').splitlines()[-120:]))
    raise RuntimeError(f'Evaluasi gagal: {return_code}; log={LOG}')

In [ ]:
import pandas as pd
from IPython.display import display
SUMMARY = OUTPUT_ROOT / 'adrian_external_summary.json'
result = json.loads(SUMMARY.read_text(encoding='utf-8'))
rows = []
for metric, values in result['aggregate'].items():
    rows.append({'metric': metric, **{key: values[key] for key in ('d0ft_mean', 'd0ft_std', 'acmc1_mean', 'acmc1_std', 'head_delta_mean', 'head_delta_std', 'head_delta_min', 'head_improved_seeds')}})
display(pd.DataFrame(rows).style.format({column: '{:.2%}' for column in ('d0ft_mean', 'd0ft_std', 'acmc1_mean', 'acmc1_std', 'head_delta_mean', 'head_delta_std', 'head_delta_min')}))
setup = result['adrian_setup']
print('ADRIAN  :', setup['images'], 'images |', setup['boxes'], 'boxes |', setup['independent_parent_ids'], 'parent IDs')
print('MISSING :', setup['classes_without_ground_truth'])
print('STATUS  :', result['directional_status'])
print('CRITERIA:', result['criteria'])
print('TRAINING:', result['training_executed'], '| TEST:', result['test_images_accessed'])
print('SUMMARY :', SUMMARY)
assert result['training_executed'] is False
assert result['test_images_accessed'] is False
assert result['further_tuning_authorized'] is False
print('Kirim tabel dan status. Ini bukti eksternal post-hoc; tidak mengubah locked-test NOT_CONFIRMED.')